# Restaurant AI - Complete Pipeline

This notebook runs the complete restaurant analytics pipeline: detection, tracking, analytics, and visualization.

In [ ]:
import sys
sys.path.append('..')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import json
from pathlib import Path

from utils.detection import PersonDetector, get_video_info
from utils.tracking import DeepSORTTracker
from utils.analytics import ZoneManager, FootfallCounter, StaffDetector, WaitTimeAnalyzer
from utils.visualization import AnalyticsEngine, draw_boxes

## Configuration

In [ ]:
VIDEO_PATH = '../data/sample_video.mp4'
MAX_FRAMES = 200
CONFIDENCE = 0.5

OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

## Initialize Components

In [ ]:
detector = PersonDetector(confidence=CONFIDENCE)
tracker = DeepSORTTracker(max_age=30, min_hits=3, iou_threshold=0.3)
zone_manager = ZoneManager()
footfall_counter = FootfallCounter(line_start=(320, 0), line_end=(320, 480))
staff_detector = StaffDetector()
wait_analyzer = WaitTimeAnalyzer()
analytics_engine = AnalyticsEngine()

zone_manager.add_zone('entrance', [(200, 400), (400, 400), (400, 450), (200, 450)])
zone_manager.add_zone('waiting', [(100, 250), (250, 250), (250, 350), (100, 350)])
zone_manager.add_zone('dining', [(400, 50), (620, 50), (620, 350), (400, 350)])

print("Components initialized!")

## Process Video

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    print("Video not found. Using camera...")
    cap = cv2.VideoCapture(0)

video_info = get_video_info(cap)
print(f"Video: {video_info}")

frame_count = 0
prev_positions = {}
track_zones = {}
staff_track_ids = set()

sample_frames = []
sample_indices = [0, 10, 20, 30, 50]

start_time = time.time()

while frame_count < MAX_FRAMES:
    ret, frame = cap.read()
    if not ret:
        break
    
    timestamp = time.time()
    
    boxes, confidences, class_ids = detector.detect(frame)
    staff_flags = staff_detector.detect_staff(frame, boxes)
    tracks, track_boxes, track_ids = tracker.update(boxes, confidences, frame, timestamp)
    
    for i, track in enumerate(tracks):
        zone = zone_manager.get_zone_at(track.center)
        if zone:
            zone_manager.update_track_zone(track.track_id, track.center, timestamp)
            if track.track_id not in track_zones:
                track_zones[track.track_id] = []
            if not track_zones[track.track_id] or track_zones[track.track_id][-1] != zone:
                track_zones[track.track_id].append(zone)
        
        if i < len(staff_flags) and staff_flags[i]:
            staff_track_ids.add(track.track_id)
    
    for track in tracks:
        if track.track_id in prev_positions:
            footfall_counter.update(track.track_id, prev_positions[track.track_id], track.center)
        prev_positions[track.track_id] = track.center
    
    if frame_count % 20 == 0:
        analytics_engine.record_footfall(timestamp, footfall_counter.entries, footfall_counter.exits)
    
    if frame_count in sample_indices:
        output = draw_boxes(frame.copy(), track_boxes, ids=track_ids)
        sample_frames.append(output)
    
    frame_count += 1
    if frame_count % 50 == 0:
        print(f"Processed {frame_count} frames, {len(tracks)} active tracks")

cap.release()
processing_time = time.time() - start_time
print(f"\nProcessed {frame_count} frames in {processing_time:.2f}s")

## Display Sample Frames

In [ ]:
for i, frame in enumerate(sample_frames):
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title(f'Sample Frame {sample_indices[i]}')
    plt.axis('off')
    plt.show()

## Results and Insights

In [ ]:
footfall_stats = footfall_counter.get_stats()
wait_stats = wait_analyzer.get_wait_time_distribution()

current_customers = footfall_stats['net']
staff_count = len(staff_track_ids)
avg_wait_time = wait_stats.get('mean', 0)

print("=" * 50)
print("STATISTICS")
print("=" * 50)
print(f"Entries: {footfall_stats['entries']}")
print(f"Exits: {footfall_stats['exits']}")
print(f"Current customers: {current_customers}")
print(f"Staff detected: {staff_count}")
print("=" * 50)

insights = analytics_engine.generate_insights(current_customers, staff_count, avg_wait_time, 0)

if insights['recommendations']:
    print("\nRecommendations:")
    for rec in insights['recommendations']:
        print(f"  - {rec}")

## Save Results

In [ ]:
analytics_engine.save_report(str(OUTPUT_DIR / 'report.json'))
analytics_engine.export_to_csv(str(OUTPUT_DIR / 'footfall.csv'))
print(f"Saved to {OUTPUT_DIR}")